**Lab type:** review  
**Course:** EDA101 — Exploratory Data Analysis  
**Lesson:** Profiling a New Dataset  
**Task:** The AI-generated profiling notebook below runs without errors, but contains three issues: one type conversion mistake, one missing check, and one missingness interpretation that is incomplete. For each issue: identify what is wrong, explain why it matters, and fix the code.

## Setup: Load the dataset

In [4]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 84320

statuses = np.random.choice(
    ['delivered', 'pending', 'shipped', 'cancelled', 'processing', 'returned'],
    size=n,
    p=[0.728, 0.129, 0.097, 0.035, 0.010, 0.0005 + 0.0005]
)

order_dates = pd.date_range('2023-01-01', periods=n, freq='1min').strftime('%Y-%m-%d').tolist()

# delivery_date is null for all pending/processing rows (~8.1% of total)
delivery_dates = [
    None if s in ('pending', 'processing') else
    (pd.Timestamp('2023-01-01') + pd.Timedelta(days=int(np.random.randint(1, 10)))).strftime('%Y-%m-%d')
    for s in statuses
]

return_flags = np.where(
    np.random.random(n) < 0.002,
    None,
    np.random.choice(['True', 'False'], n, p=[0.05, 0.95])
)

df = pd.DataFrame({
    'order_id':      ['ORD-{:05d}'.format(i) for i in range(1, n + 1)],
    'customer_id':   ['CUST-{:05d}'.format(np.random.randint(1, 31483)) for _ in range(n)],
    'order_date':    order_dates,
    'product_id':    ['PROD-{:03d}'.format(np.random.randint(1, 848)) for _ in range(n)],
    'quantity':      np.random.choice([1,2,3,4,5], n, p=[0.4,0.3,0.15,0.1,0.05]).astype(int),
    'unit_price':    np.round(np.random.choice([18.99, 29.99, 39.99, 49.99, 69.99, 99.99], n), 2),
    'discount':      np.round(np.random.choice([0.0, 0.05, 0.10, 0.15, 0.20, np.nan], n, p=[0.1,0.2,0.3,0.2,0.17,0.03]), 2),
    'shipping_cost': np.round(np.random.uniform(0, 25, n), 2),
    'region':        np.random.choice(['North', 'South', 'East', 'West', 'Central', 'Pacific', 'Mountain', 'Northeast'], n),
    'channel':       np.random.choice(['web', 'mobile', 'in-store', 'phone'], n, p=[0.45, 0.35, 0.15, 0.05]),
    'status':        statuses,
    'delivery_date': delivery_dates,
    'return_flag':   return_flags,
    'revenue':       np.round(np.random.uniform(0, 500, n), 2),
})

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

Shape: (84320, 14)

Columns: ['order_id', 'customer_id', 'order_date', 'product_id', 'quantity', 'unit_price', 'discount', 'shipping_cost', 'region', 'channel', 'status', 'delivery_date', 'return_flag', 'revenue']


---

## AI-generated profiling notebook

The cells below were generated by an AI assistant given the column names and a prompt to "profile this dataset". The code runs without errors. Your job is to find the three issues.

### Step 1 — Shape and column types

In [5]:
# AI-generated
print("Shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)

Shape: (84320, 14)

Column types:
order_id          object
customer_id       object
order_date        object
product_id        object
quantity           int64
unit_price       float64
discount         float64
shipping_cost    float64
region            object
channel           object
status            object
delivery_date     object
return_flag       object
revenue          float64
dtype: object


### Step 2 — Missing values

In [6]:
# AI-generated
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(1)
summary = pd.DataFrame({'count': missing, 'pct': missing_pct})
print(summary[summary['count'] > 0])

               count   pct
discount        2593   3.1
delivery_date  11747  13.9
return_flag      156   0.2


### Step 3 — Numeric summaries

In [7]:
# AI-generated
print(df.describe())

           quantity    unit_price      discount  shipping_cost       revenue
count  84320.000000  84320.000000  81727.000000   84320.000000  84320.000000
mean       2.100557     51.523954      0.106784      12.540101    250.639676
std        1.175219     26.914124      0.061295       7.200677    144.641060
min        1.000000     18.990000      0.000000       0.000000      0.000000
25%        1.000000     29.990000      0.050000       6.340000    124.880000
50%        2.000000     49.990000      0.100000      12.540000    251.730000
75%        3.000000     69.990000      0.150000      18.760000    375.722500
max        5.000000     99.990000      0.200000      25.000000    499.980000


---

## Issue 1: Type conversion deferred

**What to find:** Look at the output of the `describe()` call above. Which columns are missing from the numeric summary that you would expect to see — or that should appear as dates rather than being silently excluded?

**Hint:** Check `df.dtypes` again. How are `order_date` and `delivery_date` stored? How is `return_flag` stored?

In [8]:
# Detect: which columns that should be datetime or bool are stored as object?
print(df[['order_date', 'delivery_date', 'return_flag']].dtypes)

order_date       object
delivery_date    object
return_flag      object
dtype: object


**Explanation:** Write your answer here — why does running `describe()` before type conversion cause you to miss information?

<details>
<summary>🔑 Reveal answer — Issue 1</summary>

**What the problem is:** `describe()` only includes columns pandas considers numeric at call time. `order_date` and `delivery_date` are stored as `object` (string) — they are silently excluded, not summarised and not flagged. `return_flag` is likewise stored as object strings (`'True'/'False'`) rather than boolean, so it disappears from the numeric profile.

**Why this causes you to miss information:** You lose visibility into date range, completeness, and plausible value boundaries for two important columns. A subsequent `.corr()` or time-based aggregation will also silently exclude them or produce wrong results.

**Correct approach:** Always run type conversion in the first block — before any call to `.describe()`, `.corr()`, or `.groupby()`. Run `.dtypes` afterwards to confirm the conversion took effect.

</details>

In [9]:
# Fix: convert types before computing any summary statistics
df['order_date'] = pd.to_datetime(df['order_date'])
df['delivery_date'] = pd.to_datetime(df['delivery_date'])
df['return_flag'] = df['return_flag'].map({'True': True, 'False': False})

print(df[['order_date', 'delivery_date', 'return_flag']].dtypes)
print("\nNumeric describe after type conversion:")
print(df.describe())

order_date       datetime64[ns]
delivery_date    datetime64[ns]
return_flag              object
dtype: object

Numeric describe after type conversion:
                          order_date      quantity    unit_price  \
count                          84320  84320.000000  84320.000000   
mean   2023-01-29 18:43:02.163187968      2.100557     51.523954   
min              2023-01-01 00:00:00      1.000000     18.990000   
25%              2023-01-15 00:00:00      1.000000     29.990000   
50%              2023-01-30 00:00:00      2.000000     49.990000   
75%              2023-02-13 00:00:00      3.000000     69.990000   
max              2023-02-28 00:00:00      5.000000     99.990000   
std                              NaN      1.175219     26.914124   

           discount  shipping_cost                  delivery_date  \
count  81727.000000   84320.000000                          72573   
mean       0.106784      12.540101  2023-01-05 23:30:43.975031808   
min        0.000000       0.0

---

## Issue 2: Cardinality check missing

**What to find:** The AI-generated notebook profiled shape, missing values, and numeric summaries — but it skipped a check on the string columns entirely. Which check is missing, and why does it matter?

**Hint:** For each `object` column, how many unique values does it have? Are any of those counts surprising?

In [10]:
# Add the missing check: unique value counts for all object columns
for col in df.select_dtypes('object').columns:
    n_unique = df[col].nunique()
    print(f"{col:<20} {n_unique:>6} unique")

order_id              84320 unique
customer_id           29254 unique
product_id              847 unique
region                    8 unique
channel                   4 unique
status                    6 unique
return_flag               2 unique


**Explanation:** Write your answer here — what would you miss about data quality if you skipped cardinality checks on string columns?

Then for any low-cardinality columns (fewer than 20 unique values), inspect the actual values:

<details>
<summary>🔑 Reveal answer — Issue 2</summary>

**What you miss without a cardinality check:** Low-cardinality string columns look fine from a shape and missing-value perspective — but their actual values may be dirty. Skipping this check means missing that `return_flag` contains four distinct strings (`'True'`, `'Yes'`, `'False'`, `'No'`) rather than two. Without counting unique values, the column appears to be a normal boolean-encoded field.

**Why it matters:** Any downstream aggregation on `return_flag` (e.g. return rate by channel) silently double-counts because `'True'` and `'Yes'` are treated as separate categories. A cardinality check surfaces this in seconds.

**Rule of thumb:** For every `object` column, print unique value counts before any analysis. Any column with fewer than ~20 unique values deserves inspection of the actual values.

</details>

In [11]:
# Inspect the actual values for low-cardinality object columns
for col in df.select_dtypes('object').columns:
    if df[col].nunique() < 20:
        print(f"\n{col}:")
        print(df[col].value_counts())


region:
region
North        10671
Pacific      10601
Northeast    10567
Mountain     10561
Central      10512
West         10509
South        10466
East         10433
Name: count, dtype: int64

channel:
channel
web         37759
mobile      29546
in-store    12800
phone        4215
Name: count, dtype: int64

status:
status
delivered     61444
pending       10890
shipped        8233
cancelled      2827
processing      857
returned         69
Name: count, dtype: int64

return_flag:
return_flag
False    79991
True      4173
Name: count, dtype: int64


---

## Issue 3: Missingness reported but not interrogated

**What to find:** The AI notebook reported that `delivery_date` has missing values. That's correct — but reporting the count is not enough. Is the missingness random, or is it structurally explained by another column?

**Hint:** Cross-tabulate `delivery_date` missingness against `status`.

In [12]:
# Detect: is delivery_date missingness explained by order status?
print(df[df['delivery_date'].isna()]['status'].value_counts())

status
pending       10890
processing      857
Name: count, dtype: int64


**Explanation:** Write your answer here — what does this cross-tab tell you that the plain missing-value count did not? How would you treat this missing data differently knowing it is structural?

<details>
<summary>🔑 Reveal answer — Issue 3</summary>

**What the cross-tab tells you:** Virtually 100% of null `delivery_date` values belong to rows where `status` is `pending` or `processing`. The plain missing-value count says "X rows are missing a delivery date" — the cross-tab says "those rows are all in-progress orders where no delivery has occurred yet."

**How to treat it differently:** This is structural missingness — the null is meaningful, not a data quality defect. Imputing a delivery date for an order that hasn't shipped would be semantically wrong. The correct treatment is to leave the nulls intact and, if needed, derive a binary `is_delivered` indicator.

**The general principle:** Always ask whether missingness is random, systematic, or structural before deciding on a treatment. A plain count never answers that question.

</details>

In [13]:
# Confirm: what percentage of pending/processing rows are missing delivery_date?
for s in ['pending', 'processing']:
    subset = df[df['status'] == s]
    pct = subset['delivery_date'].isna().mean() * 100
    print(f"status='{s}': {pct:.1f}% missing delivery_date ({subset['delivery_date'].isna().sum()} of {len(subset)} rows)")

status='pending': 100.0% missing delivery_date (10890 of 10890 rows)
status='processing': 100.0% missing delivery_date (857 of 857 rows)


---

## Summary check

Run the corrected profiling pass in full — with type conversion first, cardinality checks included, and missingness interrogated:

In [14]:
def profile(df):
    print("=== Shape ===")
    print(df.shape)

    print("\n=== Types ===")
    print(df.dtypes)

    print("\n=== Missing values ===")
    missing = df.isna().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    print(pd.DataFrame({'count': missing, 'pct': missing_pct})[missing > 0])

    print("\n=== Cardinality (object columns) ===")
    for col in df.select_dtypes('object').columns:
        print(f"  {col:<20} {df[col].nunique():>6} unique")

    print("\n=== Numeric summary ===")
    print(df.describe())

profile(df)

=== Shape ===
(84320, 14)

=== Types ===
order_id                 object
customer_id              object
order_date       datetime64[ns]
product_id               object
quantity                  int64
unit_price              float64
discount                float64
shipping_cost           float64
region                   object
channel                  object
status                   object
delivery_date    datetime64[ns]
return_flag              object
revenue                 float64
dtype: object

=== Missing values ===
               count   pct
discount        2593   3.1
delivery_date  11747  13.9
return_flag      156   0.2

=== Cardinality (object columns) ===
  order_id              84320 unique
  customer_id           29254 unique
  product_id              847 unique
  region                    8 unique
  channel                   4 unique
  status                    6 unique
  return_flag               2 unique

=== Numeric summary ===
                          order_date      q

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Type conversion first:** `describe()` silently excludes `object`-typed columns — converting `order_date`, `delivery_date`, and `return_flag` before any summary ensures all columns appear in the profile.

2. **Cardinality check:** Skipping unique-value counts hides encoding problems; `return_flag` contains four string values (`'True'`, `'Yes'`, `'False'`, `'No'`) that inflate return-rate aggregations.

3. **Structural missingness:** The null `delivery_date` values are 100% explained by `status = pending/processing` — they are not random gaps and should not be imputed.

</details>